# Vault Collection — geoid-only retrieval

**Use case**: a collection where features are persisted but unreachable without the geoid returned at write time. The geoid acts as a capability token. No filter search, no listing, no spatial query, no tiles, no STAC `/search` — losing the geoid means losing the feature.

## Routing shape

| Operation | Driver(s) | Purpose |
|---|---|---|
| **WRITE** | `items_postgresql_driver` (sync, fatal) + `items_elasticsearch_private_driver` (sync, warn) | PG = authoritative geometry; ES = geoid-keyed index for fast lookup |
| **READ** | `items_postgresql_driver` (`hints={"geometry_exact"}`) | Authoritative un-simplified geometry |
| **SEARCH** | `items_elasticsearch_private_driver` | Geoid lookup only — driver capabilities exclude `SPATIAL_FILTER`/`ATTRIBUTE_FILTER`/`FULLTEXT` |

## Two retrieval modes

1. **Always-available — accurate geometry**: `GET /features/catalogs/{cat}/collections/{coll}/items/{geoid}` → READ driver = PG with `geometry_exact` hint → un-simplified. This is the OGC API Features standard endpoint; it's mounted whenever writes are. The writer must remember `(catalog_id, collection_id, geoid)`.
2. **Optional — fast cross-collection lookup**: `GET /search/catalogs/{cat}/geoid/{geoid}` → ES private index. Only available when the deployment includes the `search` extension (`dynastore[extension_search]`, part of `catalog_routes_grp`). Caller only needs `(catalog_id, geoid)` because the response carries `collection_id`. Returns `simplification_factor`/`simplification_mode` so the caller can decide whether to fall back to mode (1) for un-simplified geometry.

The notebook probes the `/search` extension at the start and degrades gracefully to mode (1) when it's absent. Mode (1) alone is sufficient for the vault contract — mode (2) is a convenience.

## Lockdown

A bundle of regex-scoped DENY policies (registered via `POST /iam/governance/policies`, partition_key = catalog) blocks every discovery surface — features list, STAC search, tiles, EDR, coverages, connected_systems. Paired ALLOW policies open exactly the geoid-lookup, items-by-id, and write paths.

## Pyodide notes

Runs in JupyterLite. The setup cell auto-fetches a sysadmin token from Keycloak when env-var tokens are absent (Pyodide has no shell environment). All HTTP via `httpx.AsyncClient`.

In [ ]:
import os
import json
import asyncio
import httpx

BASE = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080").rstrip("/")
KEYCLOAK = "http://localhost:8180/realms/geoid/protocol/openid-connect/token"
CATALOG_ID = "demo-vault"
COLLECTION_ID = "vault-locations"

TOKEN = (
    os.environ.get("DYNASTORE_SYSADMIN_TOKEN")
    or os.environ.get("DYNASTORE_ADMIN_TOKEN")
    or os.environ.get("DYNASTORE_TOKEN")
    or ""
)
if not TOKEN:
    print("No env token — fetching one from Keycloak (testadmin)…")
    async with httpx.AsyncClient(timeout=15.0) as kc:
        r = await kc.post(KEYCLOAK, data={
            "grant_type": "password",
            "client_id": "geoid-api",
            "client_secret": "geoid-api-secret",
            "username": "testadmin",
            "password": "testpassword",
        })
    if r.status_code == 200:
        TOKEN = r.json().get("access_token", "")
        print(f"Got token from Keycloak (len={len(TOKEN)})")
    else:
        print(f"Keycloak token request failed: {r.status_code} — {r.text[:200]}")

headers = {"Content-Type": "application/json"}
if TOKEN:
    headers["Authorization"] = f"Bearer {TOKEN}"
client = httpx.AsyncClient(base_url=BASE, headers=headers, timeout=120.0)


def _check(r, label=""):
    status = r.status_code
    ok = status < 400
    body = ""
    try:
        body = r.text[:300]
    except Exception:
        pass
    print(f"{label + ': ' if label else ''}{status}{'' if ok else f' — {body}'}")
    return r


# Probe whether the `search` extension is mounted on this deployment.
# `/search/catalogs/{any}/geoid/{any}` is the canonical fast-lookup route
# but it's only available when SCOPE includes `search`. Without it, the
# notebook uses the always-available items-by-id path instead.
try:
    _probe = await client.get("/search/catalogs/_probe_/geoid/00000000-0000-0000-0000-000000000000")
    SEARCH_AVAILABLE = _probe.status_code in (200, 400, 404)  # any answer means the route is mounted
except Exception:
    SEARCH_AVAILABLE = False

print(f"BASE             = {BASE}")
print(f"TOKEN            = {'set (' + str(len(TOKEN)) + ' chars)' if TOKEN else 'EMPTY (writes will fail)'}")
print(f"SEARCH ext       = {'mounted (fast-lookup available)' if SEARCH_AVAILABLE else 'absent (will use items-by-id only)'}")

## 1. Create catalog + vault collection (clean slate)

Vault collections live in their own catalog. Tile / EDR / connected_systems routes are **catalog-scoped**, not collection-scoped — keeping vault collections in a dedicated catalog lets the catalog-wide DENY policies (registered in step 3) cover those leak surfaces without affecting unrelated public collections.

In [ ]:
_cleanup = await client.delete(f"/stac/catalogs/{CATALOG_ID}", params={"force": "true"}, timeout=60.0)
if _cleanup.status_code not in (204, 404):
    print(f"WARN: cleanup returned {_cleanup.status_code}: {_cleanup.text[:200]}")

r = await client.post("/stac/catalogs", json={
    "id": CATALOG_ID,
    "title": "Vault Demo",
    "description": "Vault collections — geoid-only retrieval, no discovery.",
})
_check(r, "Catalog")

for _ in range(60):
    r = await client.get(f"/stac/catalogs/{CATALOG_ID}")
    if r.status_code == 200 and r.json().get("provisioning_status") == "ready":
        print("Catalog ready")
        break
    await asyncio.sleep(1)
else:
    print("WARN: catalog still provisioning after 60s")

r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections", json={
    "id": COLLECTION_ID,
    "title": "Vault Locations",
    "description": "Sensitive geometries reachable only by geoid.",
    "extent": {
        "spatial": {"bbox": [[-180, -90, 180, 90]]},
        "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]},
    },
})
_check(r, "Collection")

## 2. PUT `collection_routing_config` — vault routing

- WRITE fan-out: PG (sync, fatal) + private ES (sync, warn). Loss of the authoritative store fails the ingest; loss of the indexer warns and fixes itself on next reindex.
- READ pinned to PG with the `geometry_exact` hint — accurate geometry on the items-by-id path.
- SEARCH pinned to private ES.

> **Important — routing waterfall**: collection-scoped routing config **merges** with catalog → platform → code defaults rather than replacing them. After the PUT, GET-back will show that SEARCH still includes other drivers (e.g., `items_postgresql_driver`, `items_duckdb_driver`) inherited from the platform default. The private ES driver's lack of `SPATIAL_FILTER`/`ATTRIBUTE_FILTER`/`FULLTEXT` capabilities means filter/spatial queries are still structurally broken at the driver layer, but **the route stays open**. **Real lockdown of SEARCH must come from IAM (step 3)** — the routing config is necessary but not sufficient by itself.

In [ ]:
routing = {
    "enabled": True,
    "operations": {
        "WRITE": [
            {"driver_id": "items_postgresql_driver",
             "on_failure": "fatal", "write_mode": "sync"},
            {"driver_id": "items_elasticsearch_private_driver",
             "on_failure": "warn",  "write_mode": "sync"},
        ],
        "READ": [
            {"driver_id": "items_postgresql_driver",
             "hints": ["geometry_exact"]},
        ],
        "SEARCH": [
            {"driver_id": "items_elasticsearch_private_driver"},
        ],
    },
}
r = await client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/plugins/collection_routing_config",
    json=routing,
)
_check(r, "PUT collection_routing_config")

r = await client.get(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/plugins/collection_routing_config",
)
ops = r.json().get("operations", {})
print("\nApplied routing.operations:")
for op, entries in ops.items():
    summary = [f"{e.get('driver_id')}({','.join(e.get('hints') or []) or '-'})" for e in (entries or [])]
    print(f"  {op:8s} {summary}")

## 3. Apply the IAM lockdown bundle (policies + role binding)

Catalog admins register policies via `POST /iam/governance/policies` (`extensions/iam/service.py:235`). `partition_key` set to the catalog id scopes the policy to that catalog. DENY short-circuits ALLOW (`policies.py:424 evaluate_access`).

**Critical**: a registered policy is only *evaluated* when it is bound to a role the caller's principal holds. Registration alone is not enforcement. The pattern below:

1. Create the policy bundle (DENYs + ALLOWs).
2. Create a `vault-{cat}` Role that contains every policy id.
3. Bind that role as a parent of the four default roles (`sysadmin`, `admin`, `user`, `anonymous`) via `POST /iam/governance/hierarchies`.

After step 3 every principal inherits the bundle. The DENYs short-circuit the broader ALLOWs that the default roles already grant.

The actions list contains HTTP-method regexes (`GET`, `POST`, …); resources are URL-path regexes.

In [ ]:
CAT = CATALOG_ID
VAULT_ROLE = f'vault-{CAT}'
DEFAULT_ROLES = ['sysadmin', 'admin', 'user', 'anonymous']

policies = [
    # --- DENY discovery surfaces ---
    {'id': f'{CAT}-vault-deny-features-list', 'effect': 'DENY', 'actions': ['GET'],
     'resources': [
         f'^/features/catalogs/{CAT}/collections/[^/]+/items/?$',
         f'^/features/catalogs/{CAT}/collections/[^/]+/queryables/?$',
     ], 'partition_key': CAT},
    {'id': f'{CAT}-vault-deny-stac-discovery', 'effect': 'DENY', 'actions': ['GET', 'POST'],
     'resources': [
         f'^/stac/catalogs/{CAT}/(search|collections-search)/?$',
         f'^/stac/catalogs/{CAT}/collections/[^/]+/items/?$',
     ], 'partition_key': CAT},
    {'id': f'{CAT}-vault-deny-records-list', 'effect': 'DENY', 'actions': ['GET'],
     'resources': [f'^/records/catalogs/{CAT}/collections/[^/]+/items/?$'],
     'partition_key': CAT},
    {'id': f'{CAT}-vault-deny-tiles', 'effect': 'DENY', 'actions': ['GET'],
     'resources': [f'^/tiles/catalogs/{CAT}(/.*)?$'],
     'partition_key': CAT},
    {'id': f'{CAT}-vault-deny-edr-coverages-misc', 'effect': 'DENY', 'actions': ['GET', 'POST'],
     'resources': [
         f'^/edr/catalogs/{CAT}(/.*)?$',
         f'^/coverages/catalogs/{CAT}(/.*)?$',
         f'^/consys/catalogs/{CAT}(/.*)?$',
         f'^/wfs/catalogs/{CAT}(/.*)?$',
         f'^/maps/catalogs/{CAT}(/.*)?$',
     ], 'partition_key': CAT},
    {'id': f'{CAT}-vault-deny-search-fanout', 'effect': 'DENY', 'actions': ['GET', 'POST'],
     'resources': [
         f'^/search/catalogs/{CAT}/items-search/?$',
         f'^/search/catalogs/{CAT}/collections-search/?$',
     ], 'partition_key': CAT},
    # --- ALLOW only the vault-permitted surfaces ---
    {'id': f'{CAT}-vault-allow-geoid-lookup', 'effect': 'ALLOW', 'actions': ['GET', 'POST'],
     'resources': [f'^/search/catalogs/{CAT}/geoid(/[^/]+)?/?$'],
     'partition_key': CAT},
    {'id': f'{CAT}-vault-allow-items-by-id', 'effect': 'ALLOW', 'actions': ['GET'],
     'resources': [f'^/features/catalogs/{CAT}/collections/[^/]+/items/[^/]+/?$'],
     'partition_key': CAT},
    {'id': f'{CAT}-vault-allow-writes', 'effect': 'ALLOW',
     'actions': ['POST', 'PUT', 'PATCH', 'DELETE'],
     'resources': [
         f'^/features/catalogs/{CAT}/collections/[^/]+/items(/[^/]+)?/?$',
         f'^/stac/catalogs/{CAT}/collections/[^/]+/items(/[^/]+)?/?$',
         f'^/configs/catalogs/{CAT}(/.*)?$',
         f'^/stac/catalogs/{CAT}(/collections(/[^/]+)?)?/?$',
     ], 'partition_key': CAT},
]

# Step 1 — register policies
for p in policies:
    await client.delete(f"/iam/governance/policies/{p['id']}")
    r = await client.post('/iam/governance/policies', json=p)
    print(f"  policy {p['effect']:5s} {p['id']:55s} -> {r.status_code}")

# Step 2 — create the vault role with all policy ids attached
policy_ids = [p['id'] for p in policies]
await client.delete(f'/iam/governance/roles/{VAULT_ROLE}')
r = await client.post('/iam/governance/roles', json={
    'name': VAULT_ROLE,
    'description': f'Vault lockdown bundle for catalog {CAT}',
    'policies': policy_ids,
})
print(f"  role create {VAULT_ROLE} -> {r.status_code}")

# Step 3 — make the vault role a parent of every default role.
# DENY policies on the parent role short-circuit ALLOWs on the children.
for child in DEFAULT_ROLES:
    await client.delete('/iam/governance/hierarchies', params={'parent': VAULT_ROLE, 'child': child})
    r = await client.post('/iam/governance/hierarchies', params={'parent': VAULT_ROLE, 'child': child})
    print(f"  hierarchy {VAULT_ROLE} -> {child}: {r.status_code}")

print()
print('Note: IAM caches effective policies per principal. If the lockdown probes in step 7')
print('return 200 on routes you expect to be 403, the cache may not have flushed yet.')
print('Re-run step 7 after a few seconds OR re-acquire your token (forces principal refresh).')

## 4. Write 2 features — small + large geometry

The large feature has hundreds of vertices. ES (the indexer) has a 10MB per-doc soft limit; the private driver simplifies geometries above the threshold via `simplify_to_fit` (`elasticsearch.py:460`). The PG store is unaffected. Step 5 reads from ES (may be simplified); step 6 reads from PG (always full).

In [ ]:
import math

# Small feature: 2 vertices. NOTE: no `id` field — let the platform assign a
# fresh UUID geoid. The capability-token model needs a system-assigned id;
# supplying our own would let an attacker guess the token by knowing our naming.
SMALL = {
    'type': 'Feature',
    'geometry': {'type': 'LineString', 'coordinates': [[0.0, 0.0], [0.1, 0.1]]},
    'properties': {'label': 'small', 'sensitivity': 'low'},
}

# Large feature: dense polygon, ~5_000 vertices around a circle. Forces ES
# simplification (private ES has a 10MB-per-doc soft limit; simplify_to_fit
# kicks in above the threshold). PG stores the un-simplified polygon.
N = 5000
ring = [
    [10.0 + 0.5 * math.cos(2 * math.pi * i / N), 20.0 + 0.5 * math.sin(2 * math.pi * i / N)]
    for i in range(N)
]
ring.append(ring[0])
LARGE = {
    'type': 'Feature',
    'geometry': {'type': 'Polygon', 'coordinates': [ring]},
    'properties': {'label': 'large', 'sensitivity': 'high', 'vertex_count': N},
}

fc = {'type': 'FeatureCollection', 'features': [SMALL, LARGE]}
r = await client.post(
    f'/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items',
    json=fc,
)
_check(r, 'Write 2 features')

body = r.json() if r.status_code < 400 else {}
ids = body.get('ids') or body.get('accepted_ids') or []
print('\nAssigned geoids (capability tokens — keep them safe!):')
for g in ids:
    print(f'  {g}')
GEOIDS = ids

## 5. Optional fast lookup via `/search/.../geoid/{geoid}` (ES path)

The geoid endpoint hits the private ES index. Returns full feature payload including `simplification_factor` and `simplification_mode`. For the small feature, `simplification_mode = "none"`. For the large feature, ES will have simplified the geometry to fit — `simplification_mode != "none"`.

This step requires the `search` extension. When it is absent the cell skips and step 6 retrieves both features from PG directly (the always-available path). The geoid is the only key either way.

In [ ]:
es_results = {}
if not SEARCH_AVAILABLE:
    print('Skipping — `search` extension is not mounted on this deployment.')
    print('Step 6 will retrieve both features from PG via items-by-id (always available).')
else:
    for g in GEOIDS:
        r = await client.get(f'/search/catalogs/{CATALOG_ID}/geoid/{g}')
        if r.status_code != 200:
            print(f'  {g}: {r.status_code} {r.text[:200]}')
            continue
        coll = r.json()
        res_list = coll.get('results') or []
        if not res_list:
            print(f'  {g}: empty result — the private ES index has no record for this geoid.')
            print(f'           Cause: items_elasticsearch_private_driver write fan-out did not produce an index entry')
            print(f'           on this stack (e.g., private indexer not enabled, index name mismatch).')
            print(f'           Step 6 (PG accurate retrieval) is unaffected — PG write always succeeds first.')
            continue
        res = res_list[0]
        geom = res.get('geometry') or {}
        coords = geom.get('coordinates') or []
        if geom.get('type') == 'Polygon' and coords:
            v = len(coords[0])
        elif geom.get('type') == 'LineString':
            v = len(coords)
        else:
            v = -1
        print(
            f'  {g}\n'
            f'    geometry.type   = {geom.get("type")}\n'
            f'    vertex_count    = {v}\n'
            f'    simplification  = mode={res.get("simplification_mode")!r} factor={res.get("simplification_factor")}\n'
            f'    collection_id   = {res.get("collection_id")!r}'
        )
        es_results[g] = res

## 6. Accurate geometry via the items-by-id round-trip (PG path, `geometry_exact` hint)

The ES result already carries `collection_id`, so we can chain a second call to `GET /features/catalogs/{cat}/collections/{coll}/items/{geoid}`. That handler resolves the READ driver via `get_driver(Operation.READ, ...)`; the routing config we PUT in step 2 baked `hints={"geometry_exact"}` into the PG entry, so the items-by-id call routes to PG and returns the un-simplified geometry.

In [ ]:
# If es_results is populated, use the collection_id from the ES record
# (the realistic flow). If it's empty (private indexer not active on this
# stack), fall back to the known COLLECTION_ID — the PG path still works
# because PG was the WRITE primary.
for g in GEOIDS:
    es_res = es_results.get(g) or {}
    coll_id = es_res.get('collection_id') or COLLECTION_ID
    r = await client.get(f'/features/catalogs/{CATALOG_ID}/collections/{coll_id}/items/{g}')
    if r.status_code != 200:
        print(f'  {g}: {r.status_code} {r.text[:200]}')
        continue
    feat = r.json()
    geom = feat.get('geometry') or {}
    coords = geom.get('coordinates') or []
    if geom.get('type') == 'Polygon' and coords:
        v_pg = len(coords[0])
    elif geom.get('type') == 'LineString':
        v_pg = len(coords)
    else:
        v_pg = -1
    es_geom = es_res.get('geometry') or {}
    es_coords = es_geom.get('coordinates') or []
    if es_geom.get('type') == 'Polygon' and es_coords:
        v_es = len(es_coords[0])
    elif es_geom.get('type') == 'LineString':
        v_es = len(es_coords)
    else:
        v_es = None
    if v_es is None:
        note = '  (no ES record on this stack — PG retrieval is the canonical path)'
    elif v_pg > v_es:
        note = f'  ← PG returned {v_pg - v_es} more vertices than ES (un-simplified)'
    else:
        note = '  (no simplification at ES — geometries match)'
    print(f'  {g}  ES={v_es if v_es is not None else "—":>5}  PG={v_pg:>5} vertices{note}')

## 7. Lockdown verification — every other surface returns 403

If any of these returns 200, the IAM bundle is incomplete for this stack. A bogus geoid should return 404 (proves the geoid endpoint is open, but the token is unknown).

In [ ]:
probes = [
    ('GET',  f'/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items',
     'features list', True),
    ('GET',  f'/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/queryables',
     'features queryables', True),
    ('POST', f'/stac/catalogs/{CATALOG_ID}/search', 'STAC search', True),
    ('POST', f'/stac/catalogs/{CATALOG_ID}/collections-search', 'STAC collections search', True),
    ('GET',  f'/tiles/catalogs/{CATALOG_ID}/0/0/0.mvt', 'tile leak', True),
    ('GET',  f'/edr/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/position?coords=POINT(0 0)',
     'EDR position', True),
    # Search-extension probes — only meaningful when the extension is mounted.
    ('GET',  f'/search/catalogs/{CATALOG_ID}/items-search', 'search items-search',
     SEARCH_AVAILABLE),
    ('GET',  f'/search/catalogs/{CATALOG_ID}/geoid/00000000-0000-0000-0000-000000000000',
     'bogus geoid', SEARCH_AVAILABLE),
]

for method, path, label, applicable in probes:
    if not applicable:
        print(f'  ---- {method:4s} {path[:78]:78s}  -> SKIP ({label}: search ext not mounted)')
        continue
    if method == 'GET':
        r = await client.get(path)
    else:
        r = await client.post(path, json={})
    got = r.status_code
    if label == 'bogus geoid':
        empty_results = False
        if got == 200:
            try:
                empty_results = len((r.json() or {}).get('results') or []) == 0
            except Exception:
                pass
        verdict = 'EXPECTED' if got == 404 or empty_results else 'UNEXPECTED'
    else:
        verdict = 'BLOCKED' if got == 403 else f'OPEN — DENY did not fire'
    print(f'  {method:4s} {path[:78]:78s}  -> {got:3d}  ({label}: {verdict})')

print()
print('Interpretation:')
print('  BLOCKED      — IAM DENY policy fired correctly (vault is locked down for this surface).')
print('  OPEN         — DENY policy registered but not evaluated for this principal. Two causes:')
print('                  (a) cache not yet flushed — re-run after a few seconds, or')
print('                  (b) sysadmin role inheritance not flowing through hierarchy on this stack —')
print('                      file as follow-up; promote auto-emit into _apply_deny_policy.')
print('  EXPECTED     — bogus-geoid behavior matches contract.')
print('  SKIP         — route is part of an extension not mounted on this deployment.')

## 8. Cleanup

In [ ]:
# Best-effort cleanup so re-running the notebook is idempotent.
for child in DEFAULT_ROLES:
    await client.delete('/iam/governance/hierarchies', params={'parent': VAULT_ROLE, 'child': child})
await client.delete(f'/iam/governance/roles/{VAULT_ROLE}')
for p in policies:
    await client.delete(f"/iam/governance/policies/{p['id']}")
r = await client.delete(f'/stac/catalogs/{CATALOG_ID}', params={'force': 'true'}, timeout=60.0)
_check(r, 'DELETE catalog')
await client.aclose()
print('Done.')

## Evaluation & recommendations

### Why IAM + routing rather than a driver flag or a new schema field

- **Driver layer is the wrong place**. A driver-side gate would have to be duplicated in PG, ES public, ES private, DuckDB, Iceberg, BigQuery — and would still not cover catalog-scoped routes (tiles, EDR, coverages) where no per-collection driver is consulted. Drivers are platform-agnostic by design (`feedback_full_driver_abstraction.md`); putting access policy there violates the SSOT.
- **A new schema field** (`discovery_mode: "vault"`) was rejected in favour of implicit detection from the routing shape. The combination `WRITE includes elasticsearch_private AND SEARCH is empty (or only elasticsearch_private)` uniquely identifies vault — no other valid combo matches. The detection logic can be promoted into `_apply_deny_policy` (`elasticsearch_private/driver.py:457`) as a follow-up so the bundle is auto-emitted on routing-config-apply.
- **IAM is the correct chokepoint**. Single enforcement point at `IamMiddleware.dispatch` (`extensions/iam/middleware.py:92`); regex DENY short-circuits ALLOW (`policies.py:424`); `partition_key` scopes the policy to the catalog so a catalog admin can self-serve.

### Cross-extension impact matrix

| Extension | Discovery routes | Covered by | Notes |
|---|---|---|---|
| Features | `/items`, `/queryables` | DENY rule 1 | Collection-scoped; precise |
| STAC     | `/search`, `/collections-search`, `/items` | DENY rule 2 | Catalog and collection scoped |
| Records  | `/items` | DENY rule 3 | Collection-scoped |
| Tiles    | `/{z}/{x}/{y}.mvt` | DENY rule 4 | **Catalog-scoped** — see deep dive below |
| EDR      | `/position`, `/area`, `/cube`, `/locations` | DENY rule 5 | Catalog-scoped |
| Coverages | `/coverage`, `/domainset`, `/rangetype` | DENY rule 5 | |
| Connected Systems | `/systems`, `/datastreams`, `/observations` | DENY rule 5 | |
| WFS / Maps | `*` | DENY rule 5 | Catalog-scoped legacy surfaces |
| Search   | `/items-search`, `/collections-search` | DENY rule 6 | Geoid lookup is the ALLOW surface |
| Search   | `/geoid/{geoid}` | ALLOW rule 1 | The canonical retrieval path |
| Features | `/items/{geoid}` | ALLOW rule 2 | Accurate-geometry round-trip |

### Tile-route deep dive

Tiles are catalog-scoped (`tiles_service.py:148`) — no `{collection_id}` path segment. The simple per-collection DENY does not exist for tiles. **This notebook adopts the operational guideline of one vault collection per catalog**, so the catalog-wide tile DENY (rule 4) blocks 100% of leaks. Same applies to EDR, coverages, connected_systems, WFS, maps. Filed follow-up: handler-side filter that skips collections whose routing shape matches the implicit-vault rule, so vault and public collections can coexist in one catalog.

### Comparison with the sibling notebooks

| Notebook | WRITE | READ | SEARCH | Discovery | Use case |
|---|---|---|---|---|---|
| `private_es_only` | private ES | private ES | private ES | open by default | Smallest footprint; no PG fallback |
| `private_es_plus_pg` | PG + private ES | PG | private ES | open by default | Two-store durability + private indexer |
| `collection_vault_geoid_only` (this) | PG + private ES | PG (`geometry_exact`) | private ES | **locked down via IAM** | Capability-token semantics; geoid is the only key |

### Recommendations (filed as follow-up issues)

1. **`/search/catalogs/{cat}/geoid/{geoid}?source=geometry_exact`** — fold the two-call accurate-retrieval pattern into a single endpoint by routing the lookup through PG when the hint is requested. Today the notebook does steps 5 + 6 separately.
2. **Auto-emit the IAM bundle in `_apply_deny_policy`** — detect the implicit-vault routing shape on routing-config-apply and register the DENY/ALLOW bundle automatically. The current notebook bundles them manually.
3. **Per-handler vault filter for tiles / EDR / coverages / maps / WFS / connected_systems** — so vault and public collections can coexist in one catalog without falling back to the dedicated-catalog guideline.
4. **HATEOAS suppression in parent listings** — vault collection IDs still appear in `list_collections_in_catalog` etc. Not a leak by itself (every operation against the collection 403s) but a tightening opportunity.

### Threat-model notes

- The **geoid is a bearer token**. Treat it like a secret — never log, transmit only over secure channels, rotate by `DELETE`+re-`POST` (a new geoid is assigned on every write).
- The bogus-geoid endpoint returning 404 (vs 403) is an intentional information-leak boundary — it confirms the endpoint is open without disclosing whether the token is valid. Acceptable since a uniform-random UUID is exhaustively unguessable.
- If a deployment also wires a public ES indexer for this catalog, vault features will leak there. The routing config is the SSOT — this notebook deliberately omits public ES, and an audit job should check every vault catalog's routing config periodically.